# Environmental impact of Artificial Intelligence
AI applications are becoming ubiquitous in society, from recommendation algorithms on streaming services and social media, to semi-autonomous vehicles, large language models and more. As this happens, the concern regarding the impact of this technology on the environment and natural resources is growing. AI developers should be aware of and able to quantify the impact of the systems they develop. In this assignment, we will focus on this often overlooked cost of AI.

###  Learning goals
By the end of this notebook, you should be able to:
1. Distinguish power (W), energy (Wh), emissions (gCO₂e), and convert between them.
2. Estimate the order of magnitude energy use of everyday devices and different computational tasks.
3. Explain why the *same* computation can have very different emissions depending on where and when it runs.
4. Recognize what is “in the black box” of cloud AI (hardware, networking, cooling, replication) and why it matters for measurement.
5. Connect AI scaling to physical and computational constraints including compute demand, power feeds, grid upgrades, cooling, water use, rather than viewing progress as only a matter of improving algorithms.
6. Explain what CodeCarbon estimates—and what it does not directly measure.
7. Identify dominated and Pareto-optimal model configurations.
8. Reason about whether a marginal accuracy gain justifies additional computation; and
9. Discuss the limits of drawing general conclusions from a small, machine-specific experiment.


#  Part 1: Fundamentals of Energy Consumption

### Power (W, kW, MW)

**Power is the rate at which energy is being used at a given moment.**

It answers the question: "How fast are we consuming electricity right now?". It is measured in **Watts (W)** and its multiples and submultiples (kW, mW...).

You can think of power like **speed**.

- 60 km/h tells you how fast you’re driving.
- 50 kW tells you how fast electricity is being used.

##### Unit conversions
- **1,000 W = 1 kW**  
- **1,000 kW = 1 MW**  
- **1,000 MW = 1 GW**  

To better understand power, it may help to know how much power is required by some common applications:
- An average working laptop: **~50 W** (0.05 kW)
- An average dishwasher: **~1800 W** (1.8 kW)
- A typical household (average instantaneous): **~1000-2000 W** (1-2 kW)


### Energy (Wh, kWh, MWh)

**Energy is power multiplied by time.**

It answers the question: "How much total electricity was consumed?". It is measured in **Watt-hour (Wh)** and its multiples and submultiples (kWh, mWh...). For common everyday applications, energy is often reported in **Kilowatt-hour (kWh)**

You can think of energy like **distance**.

If you drive at 60 km/h for 2 hours:
- Distance = 120 km.

If a dishwasher draws 1.8 kW for a two-hour cycle:
- Energy = 1.8 kW × 2 hours = 3.6 kWh.

Electricity bills usually report total energy consumed (kWh) over the billing period, not how intense usage was at any one moment.

#### Unit conversions
- **1,000 Wh = 1 kWh**  
- **1,000 kWh = 1 MWh**  
- **1,000 MWh = 1 GWh**  
- **1,000 GWh = 1 TWh**  

#### Examples
- Laptop at 0.05 kW for **8 hours** → 0.4 kWh 
- Laptop at 0.05 kW for **24 hours** → 1.2 kWh

Notice how the time span changes the result.


### What about AI applications?

Artificial Intelligence has been developed in different forms using a variety of different algorithms, models, and systems. When talking about the energy required to power AI applications, it may help to have an idea of their scale and needs. The table below offers a summary of the energy scale required by well-known AI applications. It is important to note that actual values may vary significantly depending on the task. The table is based largely on Luccioni, Jernite & Strubell's paper, ["Power Hungry Processing: Watts Driving the Cost of AI Deployment?"](https://arxiv.org/html/2311.16863v3) and other reports from AI developers such as [Google](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference). Please note that actual values vary drastically by model, hardware, output length, batching, utilization, and system boundary - the values in this table should be considered only as illustrative orders of magnitude. Tasks like complex reasoning and agentic workflow are especially hard to estimate as they can involve an arbitrary number of model calls and external operations.


| Application                        | Example                                                                                          |     Typical energy scale |
| ---------------------------------- | ------------------------------------------------------------------------------------------------ | -----------------------: |
| Classification / simple prediction | Classifying an email as spam or not spam                                                         |                  **mWh** |
| Image classification               | Identifying whether an image contains a cat                                                      |      **mWh–tens of mWh** |
| Short generative-AI request        | Asking an LLM a simple question and receiving a short answer                                     |         **tenths of Wh** |
| Long text generation               | Generating or summarizing several pages of text                                                  |                  **~Wh** |
| Image generation                   | Generating a high-quality image from a text prompt                                               |           **several Wh** |
| Complex reasoning                  | Asking an LLM to solve a difficult multi-step reasoning problem                                  |   **several–tens of Wh** |
| Agentic / multi-step workflow      | An AI agent researching a topic, visiting sources, analyzing information, and producing a report | **tens–hundreds+ of Wh** |


AI applications run on physical computing hardware, ranging from small processors in individual devices to large clusters of specialized processors housed in data centers. Simple applications, such as classification using small models, may require relatively little computing power and can sometimes run directly on laptops, phones, or other low-power devices. More computationally demanding applications—particularly large language models, image generation, complex reasoning, and agentic AI—typically rely on specialized accelerators such as **GPUs (Graphics Processing Units)** or **TPUs (Tensor Processing Units),** hosted in data centers. These processors are installed in **servers**: computers designed to provide computing resources and services to other computers over a network. A server used for AI may contain several GPUs, together with CPUs, memory, storage, and networking hardware. Finally, data centers organize multiple servers into **racks**, which provide a standardized way of housing and connecting computing equipment. A single rack may therefore contain several AI servers and dozens of GPUs, supported by high-bandwidth networking (e.g., InfiniBand), advanced cooling systems, and high-capacity power feeds.

The power requirement of an AI GPU rack is estimated to be around **40–140 kW**, depending on factors such as GPU density, generation, and configuration. High-density AI GPU racks are commonly used for training and hosting large LLMs, image generation, and complex reasoning applications. This is much higher than the power required by a typical data center rack, estimated at around **5–10 kW**. This difference arises because typical data center racks primarily contain general-purpose CPU servers, storage, and networking equipment, sufficient to support workloads such as websites, databases, business applications, storage, and cloud services, but not advanced generative models.

Recall that a typical home averages ~1-2 kW. That means that **a single 50 kW AI GPU rack requires the same power of 25-50 homes running simultaneously**. Modern AI data centers contain *dozens* of GPU racks, as well as traditional CPU racks for general cloud services (email, web hosting, databases). A medium scale facility can consume 1MW (equivalent to 500-1,000 homes), while a 100 MW hyperscale data center requires the same power as 50,000-100,000 homes.

The plot below illustrates this comparison:

In [ ]:
import matplotlib.pyplot as plt

devices = ["Laptop (0.05 kW)", "Home (1.5 kW)", "GPU Rack (50 kW)"]
power_values = [0.05, 1.5, 50]

plt.figure()
plt.bar(devices, power_values)
plt.ylabel("Power (kW)")
plt.title("Order-of-Magnitude Power Comparison")
plt.xticks(rotation=30)
plt.show()

# Notice the logarithmic difference. The GPU rack is not just “bigger” it’s 3 orders of magnitude larger than a laptop.
# And 40 kW is at the lower end of the required power range.

#### <span style="color: red;">Question 1</span>

Complete the table below to see how much energy is consumed by these different users over a day or a month, assuming continuous use. The power values were chosen as representative average values for each user.

| Device    | Power (kW) | Energy in 24h (kWh) |  Energy in a month (30 days) (kWh) |  
|-----------|-----|-------------|-------------|
| Laptop    | 0.05  |    |    |
| House     | 1.5   |    |    |
| CPU rack  | 7    |    |    |
| GPU rack  | 100   |    |    |

### Emissions (gCO₂e, kgCO₂e)

One of the main concerns related to power consumption is the amount of greenhouse gases released into the atmosphere to generate the electricity being consumed. We will see later in the exercise that the climate impact of electricity use can vary substantially depending on how that electricity is produced. **CO₂e (carbon dioxide equivalent)** is a standard unit used to measure the total climate impact of all greenhouse gases (GHGs) combined, converting non-CO₂ gases into the equivalent amount of CO₂ based on their global warming potential. It simplifies emissions reporting by aggregating gases like methane and nitrous oxide into one metric. It is typically reported with its mass unit (g, kg). The table below provides a sense of what emissions at different scales represent. These comparisons are approximate and can vary depending on how products and activities are measured.

|            Approximate emissions | Example                                                                                |
| -------------------------------: | -------------------------------------------------------------------------------------- |
|                 **~0.1 kg CO₂e** | Driving an average gasoline car for roughly **0.6 km**                                 |
|                    **1 kg CO₂e** | Producing **1 kg of peas**                                                             |
|                  **2.3 kg CO₂e** | Burning **1 litre of gasoline**                                                        |
|                 **~100 kg CO₂e** | Producing **1 kg of beef**                                                             |
|                 **~120 kg CO₂e** | Flying about **500 km** on a domestic flight                                           |
|              **4.3 tonnes CO₂e** | Driving an average gasoline passenger vehicle for **one year**                         |
|     **~400,000 tonnes CO₂/year** | Annual emissions from a **natural-gas-fired power plant** (~100 MW operating continuously) |
| **~3 million tonnes CO₂e/year**  | Annual emissions from a **large coal-fired power plant** (~500 MW operating continuously) |
|**~17.3 million tonnes CO₂e/year**| Annual **community-wide emissions of the entire Metro Vancouver area**                 |

Sources:
- https://www.epa.gov/energy/greenhouse-gas-equivalencies-calculator-calculations-and-references
- https://metrovancouver.org/services/regional-planning/metro-2050-performance-monitoring/goal-3/greenhouse-gas-emission-reduction/greenhouse-gas-emissions

As you can already deduce from the table, producing electricity produces CO₂e, but the climate impact caused by consuming electricity changes depending on how it was produced. In fact, emissions related to energy production depend on many factors, including:

- Geographic location
- Time of day
- Grid mix (coal, gas, nuclear, hydro, wind, solar)
- Marginal generator on the grid*

*When demand increases, the electricity that “ramps up” to meet that extra demand is often not the *average* resource - it’s the **marginal generator** (the one that adjusts output).

Example:
- If hydro is already maxed out, and you add demand at 6pm, the *extra* electricity might come from natural gas peakers.  
- That means your “extra kWh” can be dirtier than the daily average suggests.

#### Example 1: Québec

According to the [Canada Energy Regulator profile for Québec](https://www.cer-rec.gc.ca/en/data-analysis/energy-markets/provincial-territorial-energy-profiles/provincial-territorial-energy-profiles-quebec.html), Québec’s electricity generation intensity is about 1.2 gCO₂e/kWh (2022 estimate).

It is low because Québec electricity is dominated by hydroelectric generation. Hydro produces electricity with extremely low operational emissions.

#### Example 2: U.S.

The U.S. grid is highly heterogeneous. Coal-heavy regions have much higher emissions per kWh than hydro- or nuclear-heavy regions. Skim the [EPA eGRID summary tables](https://www.epa.gov/system/files/documents/2024-01/egrid2022_summary_tables.pdf) to see the state-level emission rates.


#### <span style="color: red;">Question 2</span>

Based on what we have seen so far, would you say that stating “my model used X kWh” is enough to answer the question “How much CO₂e did it produce as a byproduct?” Why or why not? Explain your answer in 2–3 sentences.

_Write your answer here._

<br>
<br>
<br>


#### <span style="color: red;">Question 3</span>

The cell below is set up to make some basic energy consumption calculations and show the emissions produced by the same ML job executed on different grids.

In [ ]:
# Power/Energy calculator utilities
def energy_kwh(power_kw: float, hours: float) -> float:
    """Energy (kWh) = Power (kW) × time (hours)."""
    return power_kw * hours

def emissions_kgco2e(energy_kwh: float, grid_intensity_g_per_kwh: float) -> float:
    """Emissions (kgCO2e) = energy (kWh) × grid intensity (g/kWh) ÷ 1000."""
    return energy_kwh * grid_intensity_g_per_kwh / 1000.0

# Scenario: same ML job, different grids
job_power_kw = 1.2   # e.g., a beefy workstation or a small GPU server averaged over the run
job_hours = 8        # one workday training run

job_kwh = energy_kwh(job_power_kw, job_hours)
print(f"Job energy: {job_kwh:.2f} kWh")

# Approximate grid intensities (gCO2e/kWh):
# Quebec (very low) — CER reports ~1.2 gCO2e/kWh for 2022 electricity generation intensity.
quebec_g = 1.2

# Virginia (high)
virginia_g = 267.6  

print("Emissions estimates:")
print(f"  Quebec:   {emissions_kgco2e(job_kwh, quebec_g):.4f} kgCO2e")
print(f"  Virginia: {emissions_kgco2e(job_kwh, virginia_g):.4f} kgCO2e")

# Change 0 to correct value to answer questions below
texas_g = 0
print(f"  Texas: {emissions_kgco2e(job_kwh, texas_g):.4f} kgCO2e")

Complete the following tasks:
1. In the EPA eGRID summary tables, look up the CO₂e emissions for the state of Texas, and replace this value in the cell above (watch out for the unit conversion!)
2. Compute how many times the Texas' emissions are larger than Québec, as a ratio.
3. Compare the State Resource Mix of Virginia and Texas; what seems to be the reason why Virginia has lower emissions than Texas?

_Write your answer here._

<br>
<br>
<br>


## <span style="color: red;">Question 4 - Exercise: Estimating the Energy and Emissions of a New AI Application</span>

A startup launches a real-time AI translation tool. The company is considering two hardware configurations capable of running the application:

- **Configuration A – High-performance GPU server:** draws **3 kW** on average during operation.
- **Configuration B – Energy-efficient GPU server:** draws **1.5 kW** on average during operation.

Assume either server runs **18 hours per day, 30 days per month**.

### Part A - Energy Use

1. What is the total monthly energy consumption (in kWh) of each hardware configuration?
2. How much energy does Configuration B save compared with Configuration A, both in **kWh and as a percentage**?
3. How many months of electricity use by an average Canadian home does one month of Configuration A's operation represent?

*(Assume an average home uses 30 kWh/day, or approximately 900 kWh/month.)*

### Part B - Emissions Comparison

The startup could operate its servers using electricity from either of the following grids:

- **Québec grid:** 1.2 gCO₂e/kWh
- **Fossil-heavy grid:** 450 gCO₂e/kWh

4. Calculate the monthly emissions (kgCO₂e) for **all four combinations**:

| Hardware configuration | Québec grid | Fossil-heavy grid |
|---|---:|---:|
| **Configuration A – 3 kW** | ? | ? |
| **Configuration B – 1.5 kW** | ? | ? |

5. How many times larger are the emissions from running **the same hardware** on the fossil-heavy grid rather than the Québec grid?
6. Compare **Configuration A in Québec** with **Configuration B on the fossil-heavy grid**. Which produces fewer emissions? What does this tell you about the relative importance of hardware efficiency and electricity source?

### Part C - Infrastructure Thinking

7. Suppose **1,000 companies** deploy Configuration A simultaneously. What is their total instantaneous power demand in MW?
8. What would the instantaneous power demand be if all 1,000 companies instead used Configuration B? How much power would be saved?

### Reflection

9. The climate impact of computation depends on several connected factors: **how much power the hardware draws, how long it operates, and the emissions intensity of the electricity grid**. Based on your calculations, explain how each factor affects the final emissions. Which change had the largest effect in this example?


_Write your answer here._

<br>
<br>
<br>


# Part 2: How much power does AI use (at scale)?

As we have seen, AI systems vary widely in their energy use. A small tabular model trained on a laptop has a very different footprint from a large language model served to millions of users. A one-time training run is also different from continuous inference, where deployed models respond to requests around the clock. For this reason, the energy impact of AI depends not only on the algorithm, but also on scale: how often it runs, what hardware it uses, where it runs, and what infrastructure supports it.

Generative AI is especially infrastructure-intensive because it relies on dense clusters of GPUs. Large models are often trained or served across many GPUs at once, sometimes across multiple racks. High-end accelerators can draw substantial power: for example, [NVIDIA](https://www.nvidia.com/en-us/data-center/h100/) lists the H100 SXM GPU with a configurable maximum thermal design power of up to 700 W, while PCIe versions are listed around 350–400 W. A single GPU is only one device, but hundreds or thousands of GPUs running together can create an industrial-scale electrical load. 

The total energy cost is not limited to computation. Large models often need fast networking because the model, data, or workload may be split across many GPUs that must exchange information constantly. Storage systems are needed for training datasets, model checkpoints, logs, and application data. High-density GPU servers also produce significant heat, requiring advanced cooling systems, sometimes including liquid cooling. In addition, data centers need backup power, transformers, uninterruptible power supplies, and other reliability systems. A useful way to capture this overhead is **Power Usage Effectiveness (PUE)**. PUE compares total facility power to the power used directly by IT equipment. A perfect PUE would be 1.0, but real data centers also use energy for cooling, power distribution, lighting, and other support systems. (source: https://journal.uptimeinstitute.com/large-data-centers-are-mostly-more-efficient-analysis-confirms/)

This means that when we estimate the energy use of AI, we should not only ask, “How much power does the model use?” A better question is: What is the total infrastructure required to support this computation at scale? That includes GPUs, networking, storage, cooling, and the electricity grid. As AI systems become more widely deployed, their environmental impact depends on both computational choices and infrastructure choices.

For more information about how AI is expected to affect the energy sector, you may read the IEA’s *Energy and AI* summary:
- https://www.iea.org/reports/energy-and-ai/executive-summary
- https://www.iea.org/reports/energy-and-ai/energy-demand-from-ai


## Physical scaling limits

Modern AI applications require increasingly high computational power as model sizes, dataset volumes, and algorithmic complexity continue to grow. Training state-of-the-art AI models can require tuning hundreds of billions of parameters, increasing the demand for high-performance GPUs and large-scale distributed computing infrastructure. The desire for larger models is driven by empirical evidence that model performance often improves with scale, including better accuracy, generalization, and the emergence of new capabilities such as reasoning and language understanding. Unfortunately, this idea clashes with the **physical limitations** of the infrastructure supporting these systems.

When someone says: “We’ll just add more GPUs.” They are thinking in software terms. Infrastructure does not scale that way.

Scaling AI requires solving physical constraints.
- **Power supply limits**: A data center can only use as much electricity as the local power grid can physically deliver to the site. For example, if a site can receive a maximum of 80 MW, a new data center cannot simply decide to use 120 MW. Increasing the available power may require new electrical infrastructure, engineering studies, regulatory approvals, and construction, which can take years. Thus, electricity cannot necessarily be provisioned as quickly as computing infrastructure can be deployed.
- **Cooling constraints:** Almost all the electricity used by computing hardware eventually becomes heat, which must be continuously removed. As more powerful GPUs are packed into a data center, very large amounts of heat are generated in a small space. This requires substantial cooling infrastructure, such as powerful air-conditioning or liquid-cooling systems. Insufficient cooling can reduce performance, damage hardware, or force systems to shut down.
- **Water:** Many cooling systems depend on large quantities of water, making local water availability and policies potential constraints.
- **Economics & politics:** Data center expansion can also be constrained by costs, land use, electricity pricing, regulation, and community opposition.


## The black boxes of AI infrastructure

Cloud providers rarely disclose the full energy cost of AI inference. And even when they do, they may do so in a way that leaves some details opaque.

Imagine someone stating that “A single AI prompt uses X Wh”. Based on what we have seen so far, this statement leaves out many details. As a follow-up, you should ask questions like:

- On what hardware?
- In what region?
- Under what workload?
- Including cooling?
- Using average or marginal emissions?

Measurement in AI infrastructure is not straightforward. When running an AI model locally, you can relatively easily measure or estimate:
- runtime,
- CPU/GPU utilization,
- hardware being used,
- energy use and associated emissions (e.g., using tools like CodeCarbon, which we'll see later).

When using cloud-hosted AI, much of the infrastructure is outside the user's direct view. You may not know:

- the exact hardware used to process your workload,
- how hardware is shared among different workloads,
- network and storage overhead,
- redundancy and replication (extra computing resources used to make a cloud service reliable),
- cooling and other data-center overhead,
- where the computation physically occurs, and therefore the electricity grid mix.

Thus, making it much more difficult to estimate the real resource and environmental costs of cloud-based AI systems.

Suggested reading:
- MIT Technology Review: https://www.technologyreview.com/2025/05/20/1116327/ai-energy-usage-climate-footprint-big-tech/

## <span style="color: red;"> Question 5 - Google's transparency attempt</span>
Google Cloud published a [methodology estimating energy, emissions, and water use for Gemini prompts](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference). The company estimates that a median Gemini Apps text prompt uses 0.24 Wh of energy and describes this as equivalent to watching TV for less than 9 seconds.

Reflect on this content and write a short critique of the information provided (100–200 words). In particular, consider the statement “One Gemini text query is equal to 9 seconds of watching TV.”

What does this comparison tell us, and what might it hide? Consider, for example, whether all prompts require the same amount of computation, what is included in the estimate, and who produced the estimate. What additional information would you want disclosed to make the estimate more transparent and trustworthy?


_Write your answer here._

<br>
<br>
<br>
<br>
<br>



# Part 3: Measuring Model Performance, Runtime, and Estimated Emissions

In earlier parts of the assignment, we considered AI energy use at the scale of data centres, electricity grids, and infrastructure planning. This exercise moves from those system-level questions to a controlled experiment that you can run on your own machine.

Jesse Dodge, a research scientist at the Allen Institute for AI, said: “When AI developers create new systems, they’re trying to push performance, accuracy, or the model’s capabilities, and they focus less on efficiency”. And it turns out that this focus on accuracy above everything else is costly. Here is a quote from a [2022 paper by Mill et al.](https://dl.acm.org/doi/pdf/10.1145/3514094.3534175):

*There is a recognised trade-off between model accuracy and energy efficiency. In fact, the relationship has been shown to be logarithmic<sup>1</sup>. That is, in order to achieve a linear improvement in accuracy, an exponentially larger model is required. A recent study [...] confirmed the existence of the accuracy-energy trade off [...] and indicated a 30-50% saving in energy for training related to a 1% reduction in accuracy.*

<sup>1</sup> In the particular settings used in the paper.

Therefore, our goal will be to answer the question: **When several models solve the same task, how do their predictive performance, runtime, estimated energy use, and estimated emissions compare?**

This is not a complete carbon audit. A production system may also involve data collection, hyperparameter search, shared hardware, storage, networking, cooling, replication, and hardware manufacture. Instead, this notebook uses a consistent local environment to study *relative* differences among a small set of model configurations.


## What is CodeCarbon?

[CodeCarbon](https://codecarbon.io/) is a Python package designed to estimate the energy consumed and greenhouse-gas emissions associated with a program's execution. It combines execution time and available hardware information with power and location-dependent carbon-intensity estimates.

The result is an **estimate**, not a direct reading from a whole-system electricity meter. It may vary with hardware support, background activity, software versions, geographic assumptions, and the electricity mix. CodeCarbon may fall back to model-based power estimates when direct hardware measurements are unavailable.

Remember from the previous sections that the same amount of energy can result from very different amounts of emissions. Accordingly, we use CodeCarbon to compare workloads executed under similar conditions, not to assign an immutable carbon footprint to an algorithm.

References:

- [CodeCarbon methodology](https://docs.codecarbon.io/3.2/explanation/methodology/)
- [CodeCarbon Python API](https://docs.codecarbon.io/3.2/tutorials/python-api/)


In [ ]:
# Install CodeCarbon if needed.
# If this fails in your environment, the notebook will still run and record NaN for emissions.
%pip install -q codecarbon

This next cell is for importing CodeCarbon, as well as other libraries we will need for this exercise.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from codecarbon import EmissionsTracker
    CODECARBON_AVAILABLE = True
    print("CodeCarbon is available.")
except Exception as exc:
    CODECARBON_AVAILABLE = False
    EmissionsTracker = None
    print("CodeCarbon is unavailable. Runtime analysis will still run.")
    print("Import error:", exc)

## Experimental data and task

We use scikit-learn's `make_moons` generator to create a nonlinear binary-classification problem. The two classes form interleaving half-circles, and added noise makes the classes overlap.

This controlled dataset is useful because a linear decision boundary cannot represent its curved structure, whereas trees, ensembles, and neural networks can. It therefore creates a plausible trade-off: flexibility may improve accuracy, but usually requires additional computation.

The dataset is deliberately simple. It is a teaching example, not evidence that any algorithm is universally more accurate or environmentally preferable.

References:

- [scikit-learn `make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html)


In [ ]:
# A reproducible nonlinear binary-classification dataset.

X, y = make_moons(
    n_samples=10_000,
    noise=0.30,
    random_state=42
)

print("Feature matrix shape:", X.shape)
print("Class counts:", np.bincount(y))

Because this dataset has only two features, we can directly visualize its structure. The curved and overlapping class regions make it difficult for a single linear decision boundary to perfectly separate the classes.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, s=8, alpha=0.5)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Nonlinear Classification Dataset")
plt.grid(True, alpha=0.3)
plt.show()

We will compare each model using three dimensions.

### 1. Predictive performance

We measure classification **accuracy** using repeated stratified cross-validation.

A single train/test split can be sensitive to the exact observations assigned to each subset. Here, we repeat 5-fold stratified cross-validation three times with different shuffle seeds. This produces 15 accuracy values per model.

We report:

- mean cross-validation accuracy;
- standard deviation of cross-validation accuracy.

### 2. Runtime

We separately measure:

- **fit time**: time required to train the model;
- **prediction time**: average time required to predict the full dataset;
- **total time**: fit time plus average prediction time.

Runtime is not a direct measurement of electricity use. However, within the same environment, it provides a useful computational-cost dimension.

### 3. Estimated energy and emissions

With CodeCarbon we can track:

- *E* = estimated energy consumption in kilowatt-hours (kWh);
- *C* = estimated carbon emissions in grams of CO₂ equivalent (gCO₂e).

Very short jobs are difficult to estimate reliably. We therefore repeat a standardized fit-and-predict workload until a minimum duration has elapsed. Each workload unit consists of **one fit and five full-dataset predictions**. CodeCarbon tracks the aggregate workload, and we divide its energy and emissions estimates by the number of completed workload units.

If a tracked workload completes $N$ model runs, the approximate per-run estimates are:

$$
\hat{E}_{\mathrm{per\ run}} =
\frac{E_{\mathrm{tracked\ workload}}}{N}
$$

and

$$
\hat{C}_{\mathrm{per\ run}} =
\frac{C_{\mathrm{tracked\ workload}}}{N}.
$$

This creates a more standardized comparison than tracking a single sub-second execution.

CodeCarbon documents `measure_power_secs` as the configurable power-measurement interval; its documented default is 15 seconds. In this experiment, we use a 1-second interval and a repeated workload to obtain a longer observation window.

Reference: https://docs.codecarbon.io/3.2/explanation/methodology/


The function below will allow us to evaluate a classification model using repeated stratified K-fold cross-validation.

In [ ]:
# Repeated cross-validation configuration

K = 5          # Cross-validation folds
REPEATS = 3    # Repeat the 5-fold procedure 3 times

def repeated_cv_scores(model, X, y, K=5, repeats=3, scoring="accuracy", seed=0):
    """
    Parameters
    ----------
    model : scikit-learn estimator
        The model or pipeline to evaluate.

    X : array-like, shape (n_samples, n_features)
        Input features.

    y : array-like, shape (n_samples,)
        Class labels.

    K : int, default=5
        Number of folds used in each cross-validation repetition.

    repeats : int, default=3
        Number of times to repeat the complete K-fold procedure.
        Each repetition uses a different shuffle seed.

    scoring : str, default="accuracy"
        The scikit-learn scoring metric used to evaluate each fold.

    seed : int, default=0
        Starting random seed. Repetition r uses seed + r.

    Returns
    -------
    numpy.ndarray
        One score for every evaluated fold. For example, 5 folds
        repeated 3 times produces 15 scores.
    """
    
    scores = []

    for r in range(repeats):
        cv = StratifiedKFold(
            n_splits=K,
            shuffle=True,
            random_state=seed + r
        )

        fold_scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring
        )

        scores.extend(fold_scores.tolist())

    return np.asarray(scores, dtype=float)


def summarize(arr):
    """
    Compute a descriptive summary of a one-dimensional collection
    of cross-validation scores.
    """
    arr = np.asarray(arr, dtype=float)

    return {
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr, ddof=1)),
        "n": int(len(arr))
    }

## Candidate model configurations

The candidates span different inductive biases and computational demands:

- **Logistic Regression** is a regularized linear baseline.
- **Random Forest (100 trees)** introduces a nonlinear tree ensemble.
- **Random Forest (500 trees)** tests whether a fivefold increase in ensemble size yields a meaningful accuracy gain.
- **Gradient Boosting (100 estimators)** builds trees sequentially to correct earlier errors.
- **MLP (32 hidden units)** is a small nonlinear neural network.
- **MLP (128 hidden units)** tests whether increased network width improves the trade-off.

These are configurations, not abstract algorithms. Hyperparameters—including `n_jobs=-1` for the forests—affect both quality and resource use. The experiment compares the configurations exactly as specified in the cell below on the current machine.


In [ ]:
models = {
    "Linear (LogReg)": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        )
    ),

    "Random Forest (100)": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=0,
        n_jobs=-1
    ),

    "Random Forest (500)": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        random_state=0,
        n_jobs=-1
    ),

    "GradBoost (100)": GradientBoostingClassifier(
        n_estimators=100,
        random_state=0
    ),

    "Small NN (MLP 32)": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(32,),
            max_iter=300,
            random_state=0
        )
    ),

    "Larger NN (MLP 128)": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128,),
            max_iter=300,
            random_state=0
        )
    ),
}

list(models.keys())

Let's start evaluating predictive performance independently of the emissions-tracking workload. Since each model receives the same repeated cross-validation structure, the reported accuracy values are directly comparable across the candidate set.


In [ ]:
acc_summary = {}

for name, model in models.items():
    scores = repeated_cv_scores(
        model,
        X,
        y,
        K=K,
        repeats=REPEATS,
        seed=0
    )

    acc_summary[name] = summarize(scores)

accuracy_df = (
    pd.DataFrame.from_dict(acc_summary, orient="index")
    .reset_index()
    .rename(columns={
        "index": "model",
        "mean": "cv_acc_mean",
        "std": "cv_acc_std",
        "n": "n_cv_scores"
    })
    .sort_values("cv_acc_mean", ascending=False)
)

accuracy_df

### <span style="color: red;"> Question 6 - Accuracy interpretation</span>

Which accuracy differences appear practically meaningful? Was the choice to move from linear model justified? How about the choice of adopting increasingly large nonlinear models?

_Write your answer here._

<br>
<br>
<br>
<br>


## Measuring a standardized workload

For each candidate and each measurement repeat, the code below:

1. creates a fresh clone so the estimator starts unfitted;
2. fits it on the full controlled dataset;
3. predicts the full dataset five times;
4. repeats steps 1–3 until the minimum measurement duration is reached;
5. tracks the aggregate workload with CodeCarbon; and
6. normalizes aggregate energy and emissions by the number of completed workload units.

The repeated predictions lengthen the measurement window, but they also define the unit being estimated. Runtime columns report one fit and the mean time for one prediction; energy and emissions columns report one fit plus five predictions. Keep these system boundaries explicit when interpreting the results.

For a cleaner comparison, close compute-intensive applications before running this section. Do not compare numbers collected on different machines as though the hardware and electricity assumptions were controlled.


In [ ]:
def measure_energy_and_emissions(func, *args, **kwargs):
    """
    Run a callable and measure wall-clock duration plus CodeCarbon estimates.

    Returns
    -------
    result
        Return value of func.
    energy_kwh : float
        Estimated aggregate energy in kWh, or NaN.
    emissions_kg : float
        Estimated aggregate emissions in kgCO2e, or NaN.
    duration_s : float
        Total wall-clock duration of the tracked call.
    """
    t0 = time.perf_counter()

    if not CODECARBON_AVAILABLE:
        result = func(*args, **kwargs)
        duration_s = time.perf_counter() - t0
        return result, np.nan, np.nan, duration_s

    tracker = EmissionsTracker(
        log_level="error",
        save_to_file=False,
        measure_power_secs=1
        allow_multiple_runs=False
    )

    energy_kwh = np.nan
    emissions_kg = np.nan

    try:
        tracker.start()
        result = func(*args, **kwargs)
    finally:
        stopped_emissions = tracker.stop()

        if stopped_emissions is not None:
            emissions_kg = float(stopped_emissions)

        final_data = getattr(
            tracker,
            "final_emissions_data",
            None
        )

        if final_data is not None:
            tracked_energy = getattr(
                final_data,
                "energy_consumed",
                None
            )

            if tracked_energy is not None:
                energy_kwh = float(tracked_energy)

            # Prefer final_emissions_data if stop() returned None.
            if np.isnan(emissions_kg):
                tracked_emissions = getattr(
                    final_data,
                    "emissions",
                    None
                )

                if tracked_emissions is not None:
                    emissions_kg = float(tracked_emissions)

    duration_s = time.perf_counter() - t0

    return result, energy_kwh, emissions_kg, duration_s

In [ ]:
def timed_fit_predict(model, X, y, n_pred_repeats=5):
    """
    Fit one model and measure fit time plus average full-dataset prediction time.
    """
    t0 = time.perf_counter()
    model.fit(X, y)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()

    for _ in range(n_pred_repeats):
        _ = model.predict(X)

    pred_s = (
        time.perf_counter() - t1
    ) / n_pred_repeats

    return fit_s, pred_s


def measured_fit_predict_workload(
    model,
    X,
    y,
    min_duration_s=5.0,
    n_pred_repeats=5
):
    """
    Repeat independent fit-and-predict model runs until the requested
    minimum workload duration has elapsed.

    Returns
    -------
    mean_fit_s : float
        Mean fit time per completed model run.
    mean_pred_s : float
        Mean prediction time per completed model run.
    n_runs : int
        Number of completed model runs in the tracked workload.
    """
    fit_times = []
    pred_times = []

    workload_start = time.perf_counter()

    while (
        time.perf_counter() - workload_start < min_duration_s
        or len(fit_times) == 0
    ):
        model_clone = clone(model)

        fit_s, pred_s = timed_fit_predict(
            model_clone,
            X,
            y,
            n_pred_repeats=n_pred_repeats
        )

        fit_times.append(fit_s)
        pred_times.append(pred_s)

    return (
        float(np.mean(fit_times)),
        float(np.mean(pred_times)),
        len(fit_times)
    )

In [ ]:
TIME_REPEATS = 3
MIN_MEASUREMENT_DURATION_S = 5.0

rows = []

for name, model in models.items():
    print(f"Measuring: {name}")

    fit_times = []
    pred_times = []
    tracked_durations = []
    workload_runs = []
    energy_kwh_per_run = []
    emissions_g_per_run = []

    for repeat_idx in range(TIME_REPEATS):
        (
            fit_s,
            pred_s,
            n_runs
        ), energy_kwh, emissions_kg, tracked_duration_s = (
            measure_energy_and_emissions(
                measured_fit_predict_workload,
                model,
                X,
                y,
                min_duration_s=MIN_MEASUREMENT_DURATION_S,
                n_pred_repeats=5
            )
        )

        fit_times.append(fit_s)
        pred_times.append(pred_s)
        tracked_durations.append(tracked_duration_s)
        workload_runs.append(n_runs)

        if np.isnan(energy_kwh):
            energy_kwh_per_run.append(np.nan)
        else:
            energy_kwh_per_run.append(
                energy_kwh / n_runs
            )

        if np.isnan(emissions_kg):
            emissions_g_per_run.append(np.nan)
        else:
            emissions_g_per_run.append(
                (emissions_kg * 1000.0) / n_runs
            )

    rows.append({
        "model": name,
        "cv_acc_mean": acc_summary[name]["mean"],
        "cv_acc_std": acc_summary[name]["std"],

        "fit_time_mean_s": float(
            np.mean(fit_times)
        ),
        "fit_time_std_s": float(
            np.std(fit_times, ddof=1)
        ),

        "pred_time_mean_s": float(
            np.mean(pred_times)
        ),
        "pred_time_std_s": float(
            np.std(pred_times, ddof=1)
        ),

        "total_time_mean_s": float(
            np.mean(
                np.asarray(fit_times)
                + np.asarray(pred_times)
            )
        ),

        "energy_mean_kWh_per_workload_unit": (
            float(np.nanmean(energy_kwh_per_run))
            if np.any(~np.isnan(energy_kwh_per_run))
            else np.nan
        ),
        "energy_std_kWh_per_workload_unit": (
            float(np.nanstd(
                energy_kwh_per_run,
                ddof=1
            ))
            if np.sum(~np.isnan(energy_kwh_per_run)) > 1
            else np.nan
        ),

        "emissions_mean_gCO2e_per_workload_unit": (
            float(np.nanmean(emissions_g_per_run))
            if np.any(~np.isnan(emissions_g_per_run))
            else np.nan
        ),
        "emissions_std_gCO2e_per_workload_unit": (
            float(np.nanstd(
                emissions_g_per_run,
                ddof=1
            ))
            if np.sum(~np.isnan(emissions_g_per_run)) > 1
            else np.nan
        ),

        "mean_completed_workload_units": float(
            np.mean(workload_runs)
        ),
        "mean_tracked_duration_s": float(
            np.mean(tracked_durations)
        )
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("cv_acc_mean", ascending=False)
    .reset_index(drop=True)
)

results_df


For each model, higher `cv_acc_mean` is better, while lower time, energy, and emissions are better.

Two system boundaries appear in the table:

- `total_time_mean_s` is the mean time for **one fit plus one prediction**;
- energy and emissions are estimated per **workload unit: one fit plus five predictions**.

The boundaries differ because repeated predictions help create a longer CodeCarbon observation window. Do not directly interpret an energy-per-workload-unit value as the energy for the runtime column.

`mean_completed_workload_units` provides a useful reasonableness check: cheaper configurations should generally complete more units during the minimum window. The standard deviations show short-run variability, not measurement accuracy.

### <span style="color: red;"> Question 7 - Choice of model</span>

Based on this additional information, which model would you recommend using for your application? Justify your answer in 2-3 sentences.

_Write your answer here._

<br>
<br>
<br>
<br>


## Pareto frontier: accuracy versus cost

A model is not automatically preferable because it has the highest accuracy or the lowest runtime. If tiny gains require substantially greater costs, the decision depends on the needs of the application.

A **Pareto frontier** is a way to identify efficient trade-offs without combining the objectives into one score.

We say that model \(B\) **dominates** model \(A\) when:

1. \(B\) has accuracy at least as high as \(A\);
2. \(B\) has cost at most as high as \(A\); and
3. \(B\) is strictly better in at least one of the two dimensions.

A model is **Pareto-optimal** or **non-dominated** when no evaluated alternative dominates it.

The set of non-dominated models forms the **Pareto frontier**.

For a model on the frontier, moving toward higher accuracy generally requires accepting higher cost. Moving toward lower cost generally requires accepting lower accuracy.

The frontier therefore does not identify one universally best model. It identifies the models that represent efficient trade-offs among the measured objectives. Models that are not Pareto-optimal are less attractive because another evaluated model achieves equal or better accuracy at equal or lower computational cost.


In [ ]:
def pareto_nondominated(df, acc_col, cost_col):
    """
    Return model names that are non-dominated when accuracy is maximized
    and cost is minimized.

    Rows with missing accuracy or cost are excluded.
    """
    valid_df = (
        df[["model", acc_col, cost_col]]
        .dropna()
        .reset_index(drop=True)
    )

    records = valid_df.to_dict("records")
    nondominated = []

    for i, candidate in enumerate(records):
        dominated = False

        for j, challenger in enumerate(records):
            if i == j:
                continue

            at_least_as_accurate = (
                challenger[acc_col]
                >= candidate[acc_col]
            )

            at_most_as_costly = (
                challenger[cost_col]
                <= candidate[cost_col]
            )

            strictly_better = (
                challenger[acc_col]
                > candidate[acc_col]
                or challenger[cost_col]
                < candidate[cost_col]
            )

            if (
                at_least_as_accurate
                and at_most_as_costly
                and strictly_better
            ):
                dominated = True
                break

        if not dominated:
            nondominated.append(candidate["model"])

    return nondominated


## Accuracy–estimated-emissions Pareto frontier

When CodeCarbon estimates are available, we can carry out the Pareto analysis with estimated gCO₂e per workload unit (one fit plus five predictions) as the cost dimension.

These estimates are machine-, time-, and assumption-dependent. The figure compares the candidate configurations under this notebook's conditions; it does not establish a fixed global footprint for any model family.

**Before running the next cell,** try to predict which configurations will lie on the emissions frontier. Then compare your prediction with the computed result. Were your predictions correct? 



In [ ]:
emissions_col = "emissions_mean_gCO2e_per_workload_unit"

if (
    CODECARBON_AVAILABLE
    and results_df[emissions_col].notna().any()
):
    nd_emissions = pareto_nondominated(
        results_df,
        acc_col="cv_acc_mean",
        cost_col=emissions_col
    )

    emissions_frontier_df = (
        results_df[
            results_df["model"].isin(nd_emissions)
        ]
        .sort_values(emissions_col)
    )

    plt.figure(figsize=(10, 7))

    plt.scatter(
        results_df[emissions_col],
        results_df["cv_acc_mean"],
        s=70,
        label="Candidate models"
    )

    plt.scatter(
        emissions_frontier_df[emissions_col],
        emissions_frontier_df["cv_acc_mean"],
        marker="D",
        s=120,
        label="Pareto-optimal"
    )

    plt.plot(
        emissions_frontier_df[emissions_col],
        emissions_frontier_df["cv_acc_mean"],
        linestyle="--"
    )

    for _, row in results_df.iterrows():
        if pd.notna(row[emissions_col]):
            plt.annotate(
                row["model"],
                (
                    row[emissions_col],
                    row["cv_acc_mean"]
                ),
                textcoords="offset points",
                xytext=(6, 6)
            )

    plt.xlabel(
        "Mean estimated emissions per workload unit (gCO₂e)"
    )
    plt.ylabel("Repeated CV accuracy (mean)")
    plt.title(
        "Accuracy–Estimated Emissions Pareto Frontier"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()

    positive_emissions = (
        results_df[emissions_col]
        .dropna()
        .gt(0)
        .all()
    )

    if positive_emissions:
        plt.xscale("log")

    plt.show()

    print("Emissions Pareto-optimal models:")
    for model_name in nd_emissions:
        print("-", model_name)

else:
    nd_emissions = []

    print(
        "CodeCarbon emissions were not available in this "
        "environment."
    )


## Interpreting the Pareto trade-off

A Pareto frontier, while not choosing a model for you, narrows the choice set. The final selection will require an application-specific judgment about the value of accuracy and the cost of computation. A low-emissions linear model may be appropriate when computational efficiency is a priority and the application can tolerate lower predictive accuracy. A nonlinear ensemble or neural-network model may provide a meaningful improvement in performance, but the improvement comes with additional computational cost, so it should be justified by the application. It is also worth noting that the first increase in model flexibility may produce a large performance improvement, but later increases in model size may require substantially more computation for only a fractional gain, or may fail to improve accuracy at all.

In summary:

- in a low-cost or high-volume system, a model near the low-emissions end of the frontier may be preferred;
- in an application where a small accuracy improvement has substantial value, a more computationally expensive frontier model may be justified;
- a dominated model is difficult to justify on the measured objectives because another candidate provides equal or better accuracy at equal or lower estimated emissions.

### <span style="color: red;">Question 8 - Understanding the Pareto frontier</span>

Which models are Pareto-optimal in the accuracy–emissions comparison? Choose one model that is not Pareto-optimal and explain why it is dominated by another model.


_Write your answer here._

<br>
<br>
<br>
<br>


## Optional extra practice: accuracy–runtime Pareto frontier

Repeat the Pareto analysis using `total_time_mean_s` as the cost dimension. This extension is optional and is not part of the required emissions analysis.

1. Call `pareto_nondominated` with accuracy as the objective to maximize and runtime as the objective to minimize.
2. Create a dataframe containing only the runtime-Pareto-optimal configurations.
3. Plot all configurations and visually distinguish the frontier.
4. Label each point and consider using a logarithmic x-axis.
5. Identify one runtime-dominated configuration and explain which configuration dominates it.
6. Compare the optional runtime frontier with the emissions frontier. If they differ, suggest why runtime and estimated emissions need not rank configurations identically.


In [ ]:
# OPTIONAL: Complete the accuracy–runtime Pareto analysis.

runtime_col = "total_time_mean_s"

nd_time = pareto_nondominated(
    results_df,
    acc_col="cv_acc_mean",
    cost_col=runtime_col,
)

# TODO: Select the rows in nd_time and sort them by runtime.
time_frontier_df = ...

# TODO: Plot all candidates and highlight the Pareto-optimal configurations.
# Hint: adapt the structure of the emissions-frontier plot above.


## Limitations and threats to validity

### Construct validity

Accuracy captures only one aspect of model quality. Depending on the application, precision, recall, calibration, robustness, fairness, latency, or task-specific utility may matter more. CodeCarbon estimates are also not equivalent to direct whole-system electricity measurements.

### Internal validity

Runtime and emissions estimates can be affected by background processes, thermal throttling, thread scheduling, model measurement order, and estimator parallelism. Repetition reduces—but does not eliminate—this noise. The cross-validation and resource measurements also use different procedures: CV estimates generalization performance, whereas the resource workload fits and predicts on the complete synthetic dataset.

### External validity

`make_moons` has two features and is not representative of modern large-scale AI workloads. Results from one machine, dataset, and set of hyperparameters do not justify universal rankings of model families.

### Incomplete system boundary

The notebook omits the costs of data acquisition, hyperparameter search, failed experiments, deployment infrastructure, networking, storage, cooling, hardware manufacture, and end-of-life impacts, all of which can have a substantial impact in real systems.

### Carbon-intensity uncertainty

Emissions depend on when and where electricity is generated and on the intensity data available to the tracker. A location-based estimate may not reflect a specific organization's electricity contract or the marginal generator responding to added demand.


## Final thoughts - From measurement to intervention

Measurement is useful when it changes a decision. What you learned in this exercise suggests several possible actions to take in your future model development:

- Present runtime, estimated energy, and emissions alongside predictive metrics.
- Select the least costly configuration that satisfies an explicit performance requirement.
- Use early stopping and pruning when additional training or search yields negligible gains, to not waste resources on low-value computation.
- Include hyperparameter search and discarded runs in computational accounting.
- Consider time and location when deploying models; flexible workloads may sometimes run when electricity is less carbon-intensive, though this changes emissions rather than the amount of computation.
- Explain with valid justifications when a small quality gain causes a large increase in resource demand.
